# Attention DQN

DQN runner using the ego-attention feature extractor.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        nested = candidate / "highway-rl-decision-making"
        if (nested / "src").exists() and (nested / "notebooks").exists():
            return nested
    raise RuntimeError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
HELPER_DIR = PROJECT_ROOT / "notebooks" / "_shared"
helper_dir_str = str(HELPER_DIR)
if helper_dir_str not in sys.path:
    sys.path.insert(0, helper_dir_str)

from dqn_notebook_utils import (
    build_dqn_args,
    build_env_config,
    evaluate_saved_model,
    load_dqn_backend,
    show_policy_panel,
    train_and_display,
)

trainer, PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_DIR, DEFAULT_DEVICE = load_dqn_backend(
    backend_module="attention_dqn",
    notebook_subdir="attention_dqn",
    results_subdir="attention_dqn",
)

## Config

In [ ]:
import os

environment_profile = "structured_baseline"
eval_environment_profile = environment_profile

environment_overrides = {
    "lanes_count": 3,
    "vehicles_count": 20,
    "duration": 40,
    "ego_spacing": 2.0,
    "vehicles_density": 1.0,
    "simulation_frequency": 15,
    "policy_frequency": 1,
    "other_vehicles_type": "highway_env.vehicle.behavior.IDMVehicle",
    "initial_lane_id": None,
    "offroad_terminal": False,
}
# Evaluation environment: choose a profile independently from training.
# Leave overrides empty unless you intentionally want to override that
# eval profile's defaults.
eval_environment_overrides = {}

observation_config = {
    "vehicles_count": 5,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
}
action_config = {
    "type": "DiscreteMetaAction",
}
reward_config = {
    "collision_reward": -1.0,
    "right_lane_reward": 0.1,
    "high_speed_reward": 0.4,
    "lane_change_reward": 0.0,
    "normalize_reward": True,
}
min_reward_speed = 20.0
max_reward_speed = 30.0
speed_config = {
    "reward_speed_range": [min_reward_speed, max_reward_speed],
}

timesteps = 20_000
n_envs = min(4, os.cpu_count() or 1)
eval_episodes = 5
seed = 42
run_name = "attention_dqn_tuned_20k"
train_freq = 4
gradient_steps = train_freq * n_envs

hyperparameters = {
    "learning_rate": 2.5e-4,
    "buffer_size": 50_000,
    "learning_starts": 2_000,
    "batch_size": 64,
    "gamma": 0.95,
    "target_update_interval": 1_000,
    "train_freq": train_freq,
    "gradient_steps": gradient_steps,
    "exploration_fraction": 0.70,
    "exploration_final_eps": 0.10,
    "progress_every": 1_000,
    "verbose": 0,
}
attention_config = {
    "features_dim": 64,
    "attention_heads": 2,
    "attention_dropout": 0.0,
    "presence_feature_idx": 0,
    "embedding_arch": "64,64",
    "net_arch": "64,64",
}

training_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
)
saved_model_eval_env_config = build_env_config(
    profile_name=eval_environment_profile,
    profile_overrides=eval_environment_overrides,
    observation=observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
)
args = build_dqn_args(
    results_dir=RESULTS_DIR,
    run_name=run_name,
    timesteps=timesteps,
    eval_episodes=eval_episodes,
    seed=seed,
    num_envs=n_envs,
    device=DEFAULT_DEVICE,
    hyperparameters=hyperparameters,
    extra=attention_config,
)

saved_model_eval_episodes = 1000
saved_model_eval_seed = seed + 10000
saved_model_eval_name = f"saved_model_eval_{saved_model_eval_episodes}_episodes"
visualization_episodes = 5
visualization_max_steps = 300
visualization_seed = seed + 20000
visualization_stochastic = False

display(pd.DataFrame({"training": pd.Series(training_env_config), "saved_eval": pd.Series(saved_model_eval_env_config)}))


## Train

In [ ]:
summary = train_and_display(
    trainer,
    args,
    training_env_config,
    label="Attention DQN",
)

## Saved-Model Evaluation

In [ ]:
saved_eval_summary = evaluate_saved_model(
    trainer,
    summary_path=RESULTS_DIR / run_name / "summary.json",
    env_config=saved_model_eval_env_config,
    episodes=saved_model_eval_episodes,
    seed=saved_model_eval_seed,
    name=saved_model_eval_name,
    label="Attention DQN",
)

## Congestion Failure Diagnostics

Run the same congestion taxonomy on saved Attention DQN policies and save both per-episode labels and per-step traces.


In [ ]:
import json
from pathlib import Path

from stable_baselines3 import DQN

from congestion_diagnostics import evaluate_congestion_diagnostics, save_congestion_diagnostics

attention_diagnostic_episodes = 100
attention_diagnostic_seed = seed + 50_000
attention_diagnostic_config = {
    "ttc_cap": 10.0,
    "front_ttc_safe": 4.0,
    "front_ttc_critical": 1.5,
    "rear_ttc_safe": 4.0,
    "rear_ttc_critical": 1.5,
    "lane_gap_safe": 12.0,
    "bad_action_margin": 0.35,
    "no_good_action_risk": 0.85,
    "wrong_lane_quality_margin": 0.18,
    "wrong_lane_lookback_steps": 6,
    "final_lookback_steps": 4,
}

attention_diagnostic_run_names = [run_name]
for candidate_run_name in [globals().get("aggressive_safety_run_name"), "attention_dqn_aggressive_ttc_safety_20k"]:
    if candidate_run_name and candidate_run_name not in attention_diagnostic_run_names:
        attention_diagnostic_run_names.append(candidate_run_name)

attention_diagnostic_outputs = []
attention_diagnostic_frames = []
for diagnostic_run_name in attention_diagnostic_run_names:
    summary_path = RESULTS_DIR / diagnostic_run_name / "summary.json"
    if not summary_path.exists():
        print(f"Skipping missing run: {summary_path}")
        continue

    saved_summary = json.loads(summary_path.read_text(encoding="utf-8"))
    diagnostic_model = DQN.load(saved_summary["model_path"], device=DEFAULT_DEVICE)
    diagnostic_env_config = saved_summary["env_config"]
    output_dir = RESULTS_DIR / diagnostic_run_name / f"congestion_diagnostics_{attention_diagnostic_episodes}_episodes"

    diagnostic_df, diagnostic_traces = evaluate_congestion_diagnostics(
        diagnostic_model,
        trainer.make_env,
        env_config=diagnostic_env_config,
        episodes=attention_diagnostic_episodes,
        seed=attention_diagnostic_seed,
        diagnostic_config=attention_diagnostic_config,
    )
    diagnostic_paths = save_congestion_diagnostics(diagnostic_df, diagnostic_traces, output_dir)
    diagnostic_df.insert(0, "run_name", diagnostic_run_name)
    attention_diagnostic_frames.append(diagnostic_df)
    attention_diagnostic_outputs.append({"run_name": diagnostic_run_name, **diagnostic_paths})

if not attention_diagnostic_frames:
    raise RuntimeError("No saved Attention DQN runs were found for congestion diagnostics.")

attention_congestion_df = pd.concat(attention_diagnostic_frames, ignore_index=True)
label_columns = [
    "collision",
    "agent_chose_badly",
    "no_good_discrete_action",
    "wrong_lane_earlier",
    "unavoidable_rear_pressure",
]
attention_label_rates = (
    attention_congestion_df.groupby("run_name")[label_columns]
    .mean()
    .mul(100.0)
    .round(2)
    .reset_index()
)
attention_collision_breakdown = (
    attention_congestion_df.groupby(["run_name", "collision_type"])
    .size()
    .reset_index(name="episodes")
)

print(json.dumps(attention_diagnostic_outputs, indent=2))
display(attention_label_rates)
display(attention_collision_breakdown)
display(attention_congestion_df.head(20))


## Aggressive TTC Safety Reward


In [ ]:
# Retrain Attention DQN with a TTC safety term tuned to preserve aggressive flow.
aggressive_safety_observation_config = {
    "vehicles_count": 10,
    "features": ["presence", "x", "y", "vx", "vy"],
    "absolute": False,
    "see_behind": True,
}
aggressive_rear_flow_config = {
    "enabled": True,
    "spawn_on_reset": True,
    "spawn_during_episode": True,
    "vehicles_per_lane": 1,
    "lanes": "ego_and_adjacent",
    "distance_range": [25.0, 70.0],
    "speed_offset_range": [2.0, 6.0],
    "absolute_speed_range": [23.0, 34.0],
    "min_lane_gap": 18.0,
    "spawn_probability": 0.35,
    "cooldown_policy_steps": 3,
    "min_ego_progress": 80.0,
    "max_extra_vehicles": 12,
}
aggressive_ttc_safety_reward_config = {
    "enabled": True,
    "ttc_safe_threshold": 3.0,
    "ttc_target": 4.5,
    "ttc_cap": 10.0,
    "low_ttc_penalty_weight": 0.45,
    "max_low_ttc_penalty": 0.7,
    "safe_ttc_bonus_weight": 0.03,
    "max_safe_ttc_bonus": 0.05,
    "lag_penalty_weight": 0.25,
    "speed_tolerance": 1.0,
    "max_lag_penalty": 1.0,
    "rear_ttc_pressure": 5.0,
    "rear_pressure_floor": 0.35,
    "flow_radius": 120.0,
    "lanes": "ego_and_adjacent",
}

aggressive_safety_run_name = "attention_dqn_aggressive_ttc_safety_20k"
aggressive_safety_training_env_config = build_env_config(
    profile_name=environment_profile,
    profile_overrides=environment_overrides,
    observation=aggressive_safety_observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    rear_flow=aggressive_rear_flow_config,
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward=aggressive_ttc_safety_reward_config,
)
aggressive_safety_eval_env_config = build_env_config(
    profile_name=eval_environment_profile,
    profile_overrides=eval_environment_overrides,
    observation=aggressive_safety_observation_config,
    action=action_config,
    reward=reward_config,
    speed=speed_config,
    rear_flow=aggressive_rear_flow_config,
    traffic_flow_reward={"enabled": False},
    safety_ttc_flow_reward=aggressive_ttc_safety_reward_config,
)
aggressive_safety_args = build_dqn_args(
    results_dir=RESULTS_DIR,
    run_name=aggressive_safety_run_name,
    timesteps=timesteps,
    eval_episodes=eval_episodes,
    seed=seed + 400,
    num_envs=n_envs,
    device=DEFAULT_DEVICE,
    hyperparameters=hyperparameters,
    extra=attention_config,
)

display(pd.DataFrame({
    "training": pd.Series(aggressive_safety_training_env_config),
    "saved_eval": pd.Series(aggressive_safety_eval_env_config),
}))

aggressive_safety_summary = train_and_display(
    trainer,
    aggressive_safety_args,
    aggressive_safety_training_env_config,
    label="Attention DQN + Aggressive TTC Safety Reward",
)
aggressive_safety_saved_eval_summary = evaluate_saved_model(
    trainer,
    summary_path=RESULTS_DIR / aggressive_safety_run_name / "summary.json",
    env_config=aggressive_safety_eval_env_config,
    episodes=saved_model_eval_episodes,
    seed=saved_model_eval_seed + 400,
    name=saved_model_eval_name,
    label="Attention DQN + Aggressive TTC Safety Reward",
)


## Policy Panel

In [ ]:
policy_panel_rows = show_policy_panel(
    trainer,
    summary_path=RESULTS_DIR / run_name / "summary.json",
    env_config=saved_model_eval_env_config,
    episodes=visualization_episodes,
    max_steps=visualization_max_steps,
    seed=visualization_seed,
    stochastic=visualization_stochastic,
)